# SLC46A1 engraftment: the whole computational pipeline, Run all

Runs every step of the engraftment analysis in order and writes the results beside the
inputs. Nothing has to be configured: the notebook finds its own project folder, checks
its inputs, and stops on the first failed precondition rather than carrying on.

What it runs, in order:

1. Zero-shot risk scoring of the primary humanising substitutions, ESM-2 t33 650M.
2. Exhaustive rescue scan, every single substitution at every position, read at the
   high-risk positions and at the measured extracellular gate positions.
3. Direct coupling analysis on human PCFT, which is evolutionary covariation and owes
   nothing to the language model.
4. The radius and policy sweep over the rescue scan, the gate pair comparison, and the
   coupling read-out at the pairs the earlier steps identified.

Expect roughly twenty to forty minutes on a T4. The molecular dynamics is not part of
this notebook: `md/systems/` holds eight CHARMM-GUI-ready structures for that, and they
are run with the separate GROMACS template.

**Before running:** Runtime, Change runtime type, T4 GPU. Then Runtime, Run all.

In [ ]:
PROJECT = ""

import glob, os, sys, json, time, hashlib, datetime, subprocess

MARKERS = ['data/pharmacophore_position_set.tsv', 'structures/7BC6_to_human.json']

def looks_like_project(p):
    return all(os.path.exists(os.path.join(p, m)) for m in MARKERS)

DRIVE_GUESSES = [
    '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment',
    '/content/drive/MyDrive/SLC46A1 engraftment',
]

cands = []
if PROJECT:
    cands.append(PROJECT)
cands += ['/content/SLC46A1 engraftment', '/content']

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as _e:
    print('Drive not mounted (%s), looking in /content only' % type(_e).__name__)
cands += DRIVE_GUESSES
cands += sorted(glob.glob('/content/drive/MyDrive/**/data/pharmacophore_position_set.tsv',
                          recursive=True))[:5]
cands += sorted(glob.glob('/content/*/data/pharmacophore_position_set.tsv'))
cands = [c[:-len('/data/pharmacophore_position_set.tsv')]
         if c.endswith('pharmacophore_position_set.tsv') else c for c in cands]

for z in sorted(glob.glob('/content/*.zip')):
    if not os.path.isdir('/content/SLC46A1 engraftment'):
        subprocess.run(['unzip', '-q', '-o', z, '-d', '/content'], check=False)
cands += sorted(glob.glob('/content/*/data/pharmacophore_position_set.tsv'))
cands = [c[:-len('/data/pharmacophore_position_set.tsv')]
         if c.endswith('pharmacophore_position_set.tsv') else c for c in cands]

PROJECT = next((c for c in cands if looks_like_project(c)), None)
if PROJECT is None:
    raise SystemExit('FATAL project folder not found. Upload the zip to /content and run '
                     'this cell again, or set PROJECT to the folder that contains data/ '
                     'and structures/. Looked in: %s' % cands)

OUT = os.path.join(PROJECT, 'runall_outputs')
os.makedirs(OUT, exist_ok=True)
RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    print(line)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))

REQUIRED = MARKERS + ['data/parker_residues.json', 'data/hfm_positions.tsv',
                      'data/zhao_gate_chicken.json', 'data/zhao2016_gate.json',
                      'data/primary_coupling.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

log('project %s' % PROJECT)
log('outputs %s' % OUT)


In [ ]:
import shutil

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Runtime, Change runtime type, '
                     'T4 GPU, then Runtime, Run all.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so there is no usable GPU.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'torch', 'transformers', 'numpy'], check=True)

import numpy as np
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install.')
log('torch %s on %s, numpy %s' % (torch.__version__, torch.cuda.get_device_name(0),
                                  np.__version__))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]

if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
if len(chicken) != 473:
    raise SystemExit('FATAL chicken native length %d, expected 473' % len(chicken))

CK2HU = {int(k): v for k, v in json.load(
    open(os.path.join(PROJECT, 'structures', '7BC6_to_human.json')))['ck2hu'].items()}
HU2CK = {v: k for k, v in CK2HU.items()}
SHA = hashlib.sha256(chicken.encode()).hexdigest()
log('sequences ok, chicken sha256 %s' % SHA[:16])

THRESHOLD = 1e-3
MODEL = 'facebook/esm2_t33_650M_UR50D'
AA20 = 'ACDEFGHIKLMNPQRSTVWY'

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pc = {k: i for i, k in enumerate(rows[0])}
primaries = []
for r in rows[1:]:
    if r[pc['differs']] != '1':
        continue
    cp, hp = int(r[pc['chicken_pos']]), int(r[pc['human_pos']])
    caa, haa = r[pc['chicken_aa']], r[pc['human_aa']]
    if chicken[cp - 1] != caa or human[hp - 1] != haa:
        raise SystemExit('FATAL position set disagrees with the sequences at chicken %d' % cp)
    primaries.append({'chicken_pos': cp, 'chicken_aa': caa,
                      'human_pos': hp, 'human_aa': haa})
if not primaries:
    raise SystemExit('FATAL no divergent positions in the position set')
log('primary humanising substitutions: %d' % len(primaries))


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()

base_ids = tok(chicken, return_tensors='pt')['input_ids']
OFF = 1

with torch.no_grad():
    probs_wt = torch.softmax(model(base_ids.cuda()).logits[0], dim=-1)

for p in primaries:
    i = p['chicken_pos'] - 1 + OFF
    masked = base_ids.clone()
    masked[0, i] = tok.mask_token_id
    with torch.no_grad():
        pr = torch.softmax(model(masked.cuda()).logits[0, i], dim=-1)
    hid = tok.convert_tokens_to_ids(p['human_aa'])
    p['p_human_wt_background'] = float(probs_wt[i, hid])
    p['p_human_masked'] = float(pr[hid])
    p['p_chicken_masked'] = float(pr[tok.convert_tokens_to_ids(p['chicken_aa'])])
    p['high_risk'] = bool(p['p_human_wt_background'] < THRESHOLD)

keys = ['chicken_pos', 'chicken_aa', 'human_pos', 'human_aa', 'p_human_wt_background',
        'p_human_masked', 'p_chicken_masked', 'high_risk']
with open(os.path.join(OUT, 'primary_substitution_risk.tsv'), 'w') as fh:
    fh.write(chr(9).join(keys) + chr(10))
    for p in primaries:
        fh.write(chr(9).join(str(p[k]) for k in keys) + chr(10))

targets = [p for p in primaries if p['high_risk']]
if not targets:
    raise SystemExit('FATAL no high-risk substitutions, so there is nothing to rescue')
log('step 1 done, high risk %d of %d: %s' % (
    len(targets), len(primaries),
    ', '.join('%s%d->%s' % (t['chicken_aa'], t['chicken_pos'], t['human_aa'])
              for t in targets)))


In [ ]:
BATCH = 48

gates = json.load(open(os.path.join(PROJECT, 'data', 'zhao_gate_chicken.json')))
gate_list = gates['gate_positions'] if isinstance(gates, dict) and 'gate_positions' in gates else gates
GATES = []
for g in gate_list:
    cp = int(g['chicken_pos'])
    if chicken[cp - 1] != g['chicken_aa']:
        raise SystemExit('FATAL gate %d is %s in sequence, %s in the file'
                         % (cp, chicken[cp - 1], g['chicken_aa']))
    GATES.append({'chicken_pos': cp, 'chicken_aa': g['chicken_aa']})
log('gate positions: %s' % ', '.join('%s%d' % (g['chicken_aa'], g['chicken_pos'])
                                     for g in GATES))

read_idx = [t['chicken_pos'] - 1 + OFF for t in targets] + \
           [g['chicken_pos'] - 1 + OFF for g in GATES]
read_tok = [tok.convert_tokens_to_ids(t['human_aa']) for t in targets] + \
           [tok.convert_tokens_to_ids(g['chicken_aa']) for g in GATES]
NT = len(targets)

variants = [(j, y) for j in range(1, len(chicken) + 1)
            for y in AA20 if y != chicken[j - 1]]
log('variants to score: %d' % len(variants))

CKPT = os.path.join(OUT, 'scan_partial.tsv')
SCHEMA = 'runall-v1-%d-%d' % (NT, len(GATES))
done = set()
if os.path.exists(CKPT):
    first = open(CKPT).readline().strip()
    if not first.endswith(SCHEMA):
        raise SystemExit('FATAL %s was written by a different configuration (%s). '
                         'Delete it and run this cell again.' % (CKPT, first[-40:]))
    for line in open(CKPT):
        p = line.split(chr(9))
        if p and p[0].isdigit():
            done.add((int(p[0]), p[1]))
    log('resuming, %d variants already scored' % len(done))

todo = [v for v in variants if v not in done]
ids0 = base_ids[0]
t0 = time.time()
with open(CKPT, 'a') as fh:
    if not done:
        fh.write(chr(9).join(['pos', 'aa']
                             + ['t_%d' % t['chicken_pos'] for t in targets]
                             + ['g_%d' % g['chicken_pos'] for g in GATES]
                             + ['#' + SCHEMA]) + chr(10))
        fh.flush()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b + BATCH]
        batch = ids0.repeat(len(chunk), 1)
        for k, (j, y) in enumerate(chunk):
            batch[k, j - 1 + OFF] = tok.convert_tokens_to_ids(y)
        with torch.no_grad():
            logits = model(batch.cuda()).logits
        sel = torch.softmax(logits[:, read_idx, :], dim=-1)
        for k, (j, y) in enumerate(chunk):
            vals = [float(sel[k, m, read_tok[m]]) for m in range(len(read_idx))]
            fh.write(chr(9).join([str(j), y] + ['%.6e' % v for v in vals]) + chr(10))
        fh.flush()
        if (b // BATCH) % 25 == 0:
            log('scored %d of %d, %.1f min' % (b + len(chunk), len(todo),
                                               (time.time() - t0) / 60))

scan = [l.rstrip(chr(10)).split(chr(9)) for l in open(CKPT)]
body = [r for r in scan[1:] if r and r[0].isdigit()]
if len(body) != len(variants):
    raise SystemExit('FATAL scan has %d rows, expected %d' % (len(body), len(variants)))

base_of = {t['chicken_pos']: t['p_human_wt_background'] for t in targets}
RS = os.path.join(OUT, 'rescue_scan.tsv')
GP = os.path.join(OUT, 'gate_perturbation.tsv')
with open(RS, 'w') as fr, open(GP, 'w') as fg:
    fr.write(chr(9).join(['rescuer_pos', 'rescuer_from', 'rescuer_to', 'target_pos',
                          'target_human_aa', 'p_target', 'p_baseline', 'fold_change',
                          'crosses_threshold', 'separation',
                          'rescuer_identical_to_human']) + chr(10))
    fg.write(chr(9).join(['variant_pos', 'variant_from', 'variant_to', 'gate_pos',
                          'gate_aa', 'p_gate_wt', 'separation']) + chr(10))
    for r in body:
        j, y = int(r[0]), r[1]
        hj = CK2HU.get(j)
        ident = bool(hj is not None and chicken[j - 1] == human[hj - 1])
        for m, t in enumerate(targets):
            p = float(r[2 + m])
            fr.write(chr(9).join([
                str(j), chicken[j - 1], y, str(t['chicken_pos']), t['human_aa'],
                '%.6e' % p, '%.6e' % base_of[t['chicken_pos']],
                '%.3f' % (p / base_of[t['chicken_pos']]),
                str(p >= THRESHOLD), str(abs(j - t['chicken_pos'])), str(ident)]) + chr(10))
        for m, g in enumerate(GATES):
            p = float(r[2 + NT + m])
            fg.write(chr(9).join([
                str(j), chicken[j - 1], y, str(g['chicken_pos']), g['chicken_aa'],
                '%.6e' % p, str(abs(j - g['chicken_pos']))]) + chr(10))
log('step 2 done, %.1f min, wrote rescue_scan.tsv and gate_perturbation.tsv'
    % ((time.time() - t0) / 60))


In [ ]:
MAX_SEQS = 30000
THETA = 0.2
PSEUDO = 0.5
AADCA = '-ACDEFGHIKLMNPQRSTVWY'
IDX = {a: i for i, a in enumerate(AADCA)}
Q = len(AADCA)
L = len(human)

A3M = os.path.join(OUT, 'Q96NT5.a3m')
if not os.path.exists(A3M) or os.path.getsize(A3M) < 10000:
    import requests
    host = 'https://api.colabfold.com'
    r = requests.post(host + '/ticket/msa',
                      data={'q': '>Q96NT5' + chr(10) + human, 'mode': 'env'}, timeout=120)
    r.raise_for_status()
    tid = r.json()['id']
    for _ in range(600):
        s = requests.get(host + '/ticket/' + tid, timeout=60).json()['status']
        if s in ('COMPLETE', 'ERROR'):
            break
        time.sleep(5)
    if s != 'COMPLETE':
        raise SystemExit('FATAL the alignment service returned %s' % s)
    import tarfile, io
    blob = requests.get(host + '/result/download/' + tid, timeout=600).content
    with tarfile.open(fileobj=io.BytesIO(blob)) as tf:
        names = [n for n in tf.getnames() if n.endswith('.a3m')]
        if not names:
            raise SystemExit('FATAL no a3m in the alignment result')
        with open(A3M, 'wb') as fh:
            fh.write(tf.extractfile(names[0]).read())
log('alignment file %d bytes' % os.path.getsize(A3M))

seqs = [l.rstrip() for l in open(A3M) if l.strip() and not l.startswith('>')]
aln = []
for s in seqs:
    kept = ''.join(c for c in s if not c.islower())
    if len(kept) == L:
        aln.append([IDX.get(c.upper(), 0) for c in kept])
A = np.array(aln, dtype=np.int8)
if A.shape[0] < 500:
    raise SystemExit('FATAL only %d usable sequences in the alignment' % A.shape[0])
if A.shape[0] > MAX_SEQS:
    sel = np.linspace(0, A.shape[0] - 1, MAX_SEQS).astype(int)
    sel[0] = 0
    A = A[sel]
B = A.shape[0]
log('alignment matrix %d x %d' % A.shape)

oh = np.zeros((B, L, Q), dtype=np.float32)
oh[np.arange(B)[:, None], np.arange(L)[None, :], A] = 1.0
flat = oh.reshape(B, L * Q)

cut = (1.0 - THETA) * L
neigh = np.zeros(B, dtype=np.int64)
for i in range(0, B, 512):
    neigh[i:i + 512] = ((flat[i:i + 512] @ flat.T) >= cut).sum(axis=1)
w = 1.0 / np.maximum(neigh, 1).astype(np.float64)
w32 = w.astype(np.float32)
wsum = w.sum()
log('effective sequences %.1f of %d' % (wsum, B))

fi = (oh * w32[:, None, None]).sum(axis=0) / wsum
fi = (1 - PSEUDO) * fi + PSEUDO / Q
cnt = (flat * w32[:, None]).T @ flat
cnt /= wsum
pair = cnt.reshape(L, Q, L, Q)[:, 1:, :, 1:]
pair = (1 - PSEUDO) * pair + PSEUDO / (Q * Q)
pair = pair.astype(np.float64)
for i in range(L):
    pair[i, :, i, :] = np.diag(fi[i, 1:])
del oh, flat, cnt

C = (pair - fi[:, 1:, None, None] * fi[None, None, :, 1:]).reshape(L * (Q - 1), L * (Q - 1))
del pair
J = -np.linalg.inv(C + 1e-6 * np.eye(C.shape[0]))
del C
J = J.reshape(L, Q - 1, L, Q - 1)
DI = np.sqrt((J ** 2).sum(axis=(1, 3)))
np.fill_diagonal(DI, 0.0)
mi = DI.mean(axis=1)
S = DI - np.outer(mi, mi) / DI.mean()
np.fill_diagonal(S, 0.0)
np.save(os.path.join(OUT, 'dca_apc_human.npy'), S)
log('step 3 done, coupling matrix %s' % (S.shape,))


In [ ]:
RADII = [3, 5, 8, 12, 20, 35, 60, None]
POLICIES = ['published', 'strict', 'annotated']

parker = json.load(open(os.path.join(PROJECT, 'data', 'parker_residues.json')))['residues']
protected = set()
for k, v in parker.items():
    cp, hp = int(k), v['human']
    if chicken[cp - 1] == human[hp - 1]:
        protected.add(cp)
unmapped = []
for l in list(open(os.path.join(PROJECT, 'data', 'hfm_positions.tsv')))[1:]:
    hp = int(l.split(chr(9))[0])
    cp = HU2CK.get(hp)
    if cp is None:
        unmapped.append(hp)
    else:
        protected.add(cp)
if unmapped:
    raise SystemExit('FATAL hereditary folate malabsorption positions with no chicken '
                     'partner: %s' % unmapped)
log('protected positions for the annotated policy: %d' % len(protected))

def allowed(policy, j, tp):
    if j == tp:
        return False
    if policy == 'published':
        return True
    hj = CK2HU.get(j)
    ident = hj is not None and chicken[j - 1] == human[hj - 1]
    if policy == 'strict':
        return not ident
    return j not in protected

rs = [l.rstrip(chr(10)).split(chr(9)) for l in open(RS)]
rc = {k: i for i, k in enumerate(rs[0])}
best = {}
for r in rs[1:]:
    best[(int(r[rc['rescuer_pos']]), r[rc['rescuer_to']], int(r[rc['target_pos']]))] = (
        float(r[rc['p_target']]), int(r[rc['separation']]))

summary = []
print(chr(10) + 'radius  policy      rescued  crossing  nearest')
for rad in RADII:
    for pol in POLICIES:
        rescued, crossing, nearest = set(), 0, None
        for (j, y, tp), (p, sep) in best.items():
            if rad is not None and sep > rad:
                continue
            if not allowed(pol, j, tp):
                continue
            if p >= THRESHOLD:
                crossing += 1
                rescued.add(tp)
                nearest = sep if nearest is None else min(nearest, sep)
        lab = 'all' if rad is None else str(rad)
        summary.append({'radius': lab, 'policy': pol, 'targets_rescued': len(rescued),
                        'rescuers_crossing': crossing,
                        'nearest_rescuer_separation': nearest,
                        'rescued_positions': sorted(rescued)})
        print('%6s  %-10s  %7d  %8d  %s' % (lab, pol, len(rescued), crossing,
                                            'none' if nearest is None else nearest))

per_target = {}
for t in targets:
    tp = t['chicken_pos']
    per_target[str(tp)] = {}
    for pol in POLICIES:
        cands = sorted(((p, j, y, sep) for (j, y, q), (p, sep) in best.items()
                        if q == tp and allowed(pol, j, tp)), reverse=True)[:5]
        per_target[str(tp)][pol] = [
            {'p_target': p, 'rescuer_pos': j, 'rescuer_to': y, 'separation': sep,
             'crosses': p >= THRESHOLD,
             'fold_change': p / base_of[tp]} for p, j, y, sep in cands]

json.dump({'threshold': THRESHOLD, 'protected': sorted(protected), 'summary': summary,
           'per_target': per_target,
           'baseline': {str(k): v for k, v in base_of.items()}},
          open(os.path.join(OUT, 'rescue_sweep_report.json'), 'w'), indent=1)
log('step 4a done, wrote rescue_sweep_report.json')


In [ ]:
import statistics

zg = json.load(open(os.path.join(PROJECT, 'data', 'zhao2016_gate.json')))
PAIRS = [(p, 'spontaneous disulfide') for p in zg.get('spontaneous_disulfide', [])] + \
        [(p, 'CuPh and cadmium induced')
         for p in zg.get('induced_by_cuph_and_cadmium', [])]

gp = [l.rstrip(chr(10)).split(chr(9)) for l in open(GP)]
gc = {k: i for i, k in enumerate(gp[0])}
by_gate = {}
for r in gp[1:]:
    by_gate.setdefault(int(r[gc['gate_pos']]), []).append(
        (int(r[gc['variant_pos']]), float(r[gc['p_gate_wt']])))

def position_score(gate, pos):
    vals = [p for j, p in by_gate[gate] if j == pos]
    return statistics.median(vals) if vals else None

gate_pct = {}
for gate in by_gate:
    scores = {}
    for pos in set(j for j, _ in by_gate[gate]):
        s = position_score(gate, pos)
        if s is not None:
            scores[pos] = s
    order = sorted(scores, key=lambda k: scores[k])
    gate_pct[gate] = {p: 100.0 * i / (len(order) - 1) for i, p in enumerate(order)}

hu_of = {g['chicken_pos']: CK2HU.get(g['chicken_pos']) for g in GATES}
ck_of = {v: k for k, v in hu_of.items() if v}

print(chr(10) + 'measured pair            kind                       sep   percentiles')
measured = []
for name, kind in PAIRS:
    a, b = [int(''.join(ch for ch in part if ch.isdigit())) for part in name.split('/')]
    ca, cb = ck_of.get(a), ck_of.get(b)
    if ca is None or cb is None:
        raise SystemExit('FATAL %s has a member outside the gate set' % name)
    pa = gate_pct[cb].get(ca)
    pb = gate_pct[ca].get(cb)
    measured.append({'pair': name, 'kind': kind, 'separation': abs(a - b),
                     'percentile_a_on_b': pa, 'percentile_b_on_a': pb})
    print('%-24s %-26s %4d   %.1f and %.1f' % (name, kind, abs(a - b), pa, pb))

S = np.load(os.path.join(OUT, 'dca_apc_human.npy'))
def coup(h1, h2):
    return float(S[h1 - 1, h2 - 1])

iu = np.triu_indices(S.shape[0], 1)
allv = np.sort(S[iu])
def pct(v):
    return 100.0 * float(np.searchsorted(allv, v)) / len(allv)

prim_hu = [p['human_pos'] for p in primaries]
prim_pairs = [(a, b, coup(a, b), pct(coup(a, b)))
              for i, a in enumerate(prim_hu) for b in prim_hu[i + 1:]]
comp_pairs = [(392, 196), (392, 305), (109, 305)]
gate_pairs = [(m['pair'], *[int(''.join(c for c in s if c.isdigit()))
                            for s in m['pair'].split('/')]) for m in measured]

print(chr(10) + 'couplings, human numbering, percentile among all pairs')
cp_out = []
for a, b in comp_pairs:
    v = coup(a, b)
    cp_out.append({'pair': '%d-%d' % (a, b), 'coupling': v, 'percentile': pct(v),
                   'kind': 'measured compensatory'})
    print('  compensatory %3d-%3d  %.4f  %.1f' % (a, b, v, pct(v)))
for name, a, b in gate_pairs:
    v = coup(a, b)
    cp_out.append({'pair': name, 'coupling': v, 'percentile': pct(v),
                   'kind': 'measured cross-link'})
    print('  cross-link %-14s %.4f  %.1f' % (name, v, pct(v)))
top_prim = sorted(prim_pairs, key=lambda x: -x[2])[:5]
for a, b, v, q in top_prim:
    print('  primaries  %3d-%3d  %.4f  %.1f' % (a, b, v, q))

json.dump({'gate_pair_comparison': measured,
           'couplings': cp_out,
           'primary_pair_couplings': [{'a': a, 'b': b, 'coupling': v, 'percentile': q}
                                      for a, b, v, q in prim_pairs],
           'chicken_sha256': SHA},
          open(os.path.join(OUT, 'coupling_and_gate_report.json'), 'w'), indent=1)
log('step 4b done, wrote coupling_and_gate_report.json')


In [ ]:
import shutil

meta = {'model': MODEL, 'threshold': THRESHOLD, 'n_primary': len(primaries),
        'n_high_risk': len(targets), 'n_variants': len(variants),
        'chicken_sha256': SHA,
        'finished_utc': datetime.datetime.utcnow().isoformat(timespec='seconds')}
json.dump(meta, open(os.path.join(OUT, 'run_meta.json'), 'w'), indent=2)

base = os.path.join('/content', 'SLC46A1_runall_outputs')
zip_path = shutil.make_archive(base, 'zip', OUT)
log('wrote %s (%d bytes)' % (zip_path, os.path.getsize(zip_path)))

for f in sorted(os.listdir(OUT)):
    print('%10d  %s' % (os.path.getsize(os.path.join(OUT, f)), f))

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('download not offered in this environment: %s' % e)
    print('the zip is at %s' % zip_path)


## What to send back

`SLC46A1_runall_outputs.zip`. It holds the risk scoring, the full rescue and gate scan,
the coupling matrix, and the three report files, with `run.log` recording every step and
`run_meta.json` carrying the sequence hash so the run can be tied to the inputs.

If a cell stops with a line beginning FATAL, that is the notebook refusing to continue on
bad input rather than a crash. The message says what is wrong.